In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [22]:
df = pd.read_csv('diabetes.csv')
df.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


In [23]:
columns_to_replace = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[columns_to_replace] = df[columns_to_replace].replace(0, np.nan)
df[columns_to_replace] = df[columns_to_replace].fillna(df[columns_to_replace].median())
df.head(10) #преобразовали данные с пропусками, заполнили медианой

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1
5,5,116.0,74.0,29.0,125.0,25.6,0.201,30,0
6,3,78.0,50.0,32.0,88.0,31.0,0.248,26,1
7,10,115.0,72.0,29.0,125.0,35.3,0.134,29,0
8,2,197.0,70.0,45.0,543.0,30.5,0.158,53,1
9,8,125.0,96.0,29.0,125.0,32.3,0.232,54,1


In [24]:
print(f"{df.shape[0]} образцов")
print(f"{df.shape[1]} признаков")
print()
print('1 - есть диабет, 0 - нет диабета')
print(df['Outcome'].value_counts())

768 образцов
9 признаков

1 - есть диабет, 0 - нет диабета
Outcome
0    500
1    268
Name: count, dtype: int64


In [27]:
def get_iqr_mask(s : pd.Series):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

mask = pd.Series(False, index=df.index)
for col in ['Glucose', 'BloodPressure', 'BMI']:
    mask |= get_iqr_mask(df[col])
new_df = df[~mask].reset_index(drop=True)
print(f"было строк: {df.shape[0]}")
print(f"стало строк: {new_df.shape[0]}")

было строк: 768
стало строк: 748


In [28]:
dataset_lenght : int = new_df.shape[0]
P_diabetes : float = new_df['Outcome'].value_counts()[1] / dataset_lenght
#P_diabetes : float = (df['Outcome'] == 1).sum() / dataset_lenght
print(f"вероятность диабета в выборке: {P_diabetes}")

P_older_30 : float = (new_df['Age'] > 30).sum() / dataset_lenght
print(f"доля людей старше 30 лет в выборке: {P_older_30}")

вероятность диабета в выборке: 0.3449197860962567
доля людей старше 30 лет в выборке: 0.45588235294117646


In [29]:
X = new_df['BMI'].tolist()

EX = sum(X) / dataset_lenght
print(f"мат. ожидание {EX}")

DX = sum(pow((Xi - EX), 2) for Xi in X) / (dataset_lenght - 1)
print(f"дисперсия {DX}")

P_more_than_35 = sum(1 for Xi in X if Xi > 35) / dataset_lenght
print(f"вероятность, что показатель BMI больше 35 ед.: {P_more_than_35}")

new_df_2 = [out for out, age in zip(df['Outcome'], df['Age']) if age > 23]
P_conditional = sum(new_df_2) / len(new_df_2)
print(f"какова вероятность, у человека старше 23 лет есть диабет {P_conditional}") 

мат. ожидание 32.143315508021374
дисперсия 40.82302085346726
вероятность, что показатель BMI больше 35 ед.: 0.3048128342245989
какова вероятность, у человека старше 23 лет есть диабет 0.4117647058823529


In [30]:
df_5 = new_df.sample(n=5, random_state=42)
df_50 = new_df.sample(n=50, random_state=42)
df_250 = new_df.sample(n=250, random_state=42)
df_500 = new_df.sample(n=500, random_state=42)

df_50.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
580,2,82.0,52.0,22.0,115.0,28.5,1.699,25,0
356,0,101.0,64.0,17.0,125.0,21.0,0.252,21,0
133,5,105.0,72.0,29.0,325.0,36.9,0.159,28,0
250,11,155.0,76.0,28.0,150.0,33.3,1.353,51,1
299,2,124.0,68.0,28.0,205.0,32.9,0.875,30,1
388,3,193.0,70.0,31.0,125.0,34.9,0.241,25,1
336,1,139.0,46.0,19.0,83.0,28.7,0.654,22,0
155,7,102.0,74.0,40.0,105.0,37.2,0.204,45,0
527,3,129.0,92.0,49.0,155.0,36.4,0.968,32,1
734,8,154.0,78.0,32.0,125.0,32.4,0.443,45,1


#### демонстрация закона больших чисел

In [31]:
X_5 = df_5['BMI'].tolist()
X_50 = df_50['BMI'].tolist()
X_250 = df_250['BMI'].tolist()
X_500 = df_500['BMI'].tolist()

EX_5 = sum(X_5) / len(X_5)
EX_50 = sum(X_50) / len(X_50)
EX_250 = sum(X_250) / len(X_250)
EX_500 = sum(X_500) / len(X_500)

print(f"мат. ожидание на выборке размером 5: {EX_5}. доля от мат. ожидания всей выборки: {EX_5/EX}")
print(f"мат. ожидание на выборке размером 50: {EX_50}. доля от мат. ожидания всей выборки: {EX_50/EX}")
print(f"мат. ожидание на выборке размером 250: {EX_250}. доля от мат. ожидания всей выборки: {EX_250/EX}")
print(f"мат. ожидание на выборке размером 500: {EX_500}. доля от мат. ожидания всей выборки: {EX_500/EX}")

мат. ожидание на выборке размером 5: 30.52. доля от мат. ожидания всей выборки: 0.9494975710387973
мат. ожидание на выборке размером 50: 31.608000000000008. доля от мат. ожидания всей выборки: 0.9833459772409671
мат. ожидание на выборке размером 250: 32.30880000000001. доля от мат. ожидания всей выборки: 1.0051483330006
мат. ожидание на выборке размером 500: 32.396799999999985. доля от мат. ожидания всей выборки: 1.007886071737539
